# Experiment 7: Point Multiplication Performance

Every FROST operation, signing, verification, DKG, depends on elliptic curve
scalar multiplication: computing k·P for a scalar k and point P. This notebook
benchmarks two strategies and explains why production implementations are
orders of magnitude faster.

**Related:** Guide Chapter 9 (Architecture)

In [ ]:
import sys
sys.path.insert(0, "../src")

import time
from frost import Scalar, Point, G, Q

# Warm up: trigger precomputed table construction
_ = 1 * G
print("Precomputed table ready (256 doublings of G cached).")

# Generate a random scalar for benchmarking
k = int(Scalar.random())
print(f"Test scalar k has {bin(k).count('1')} set bits out of {k.bit_length()} total bits")


## Naive Double-and-Add

The standard algorithm scans the scalar bit by bit. For each bit:
- **Double** the running point (shift to the next bit position)
- **Add** if the bit is set

For a 256-bit scalar, this requires 256 doublings + ~128 additions (on average,
half the bits are set in a random scalar).


In [ ]:
# Use a non-generator point to bypass the precomputed table
# (the fast path only activates for the exact generator point G)
P = G + G  # 2·G, a different point

trials = 3
times = []
for _ in range(trials):
    start = time.time()
    result_naive = k * P
    elapsed = time.time() - start
    times.append(elapsed)

naive_time = min(times)
print(f"Naive double-and-add ({trials} trials, best): {naive_time:.3f}s")
print(f"  256 doublings + {bin(k).count('1')} additions = {256 + bin(k).count('1')} point operations")


## Precomputed Table (Generator Point)

For the generator point G specifically, we precompute a table of 2ⁱ·G for
i = 0..255. Then scalar multiplication only needs additions (for set bits).
No doublings at all, they're already baked into the table.


In [ ]:
trials = 3
times = []
for _ in range(trials):
    start = time.time()
    result_fast = k * G
    elapsed = time.time() - start
    times.append(elapsed)

fast_time = min(times)
print(f"Precomputed table ({trials} trials, best): {fast_time:.3f}s")
print(f"  0 doublings + {bin(k).count('1')} additions = {bin(k).count('1')} point operations")


## Comparison


In [ ]:
speedup = naive_time / fast_time if fast_time > 0 else float('inf')
set_bits = bin(k).count('1')

print(f"{'Method':<25} {'Time (s)':<12} {'Point Ops':<12} {'Doublings':<12} {'Additions'}")
print("-" * 73)
print(f"{'Naive double-and-add':<25} {naive_time:<12.3f} {256 + set_bits:<12} {256:<12} {set_bits}")
print(f"{'Precomputed table':<25} {fast_time:<12.3f} {set_bits:<12} {0:<12} {set_bits}")
print(f"\nSpeedup: {speedup:.1f}x")
print(f"\nThe precomputed table eliminates all 256 doublings, keeping only")
print(f"the ~{set_bits} additions (one per set bit in the scalar).")


## Correctness Check

We verify correctness using linearity: k·(2G) computed naively should equal
k·G + k·G computed via the precomputed table, since k·(2G) = 2·(k·G).

In [ ]:
# k * (2G) via naive == k*G + k*G via precomputed?
print(f"Naive k*(2G) == fast k*G + fast k*G: {result_naive == result_fast + result_fast}")
print(f"Both methods compute correct results.")


## What Production Implementations Do Differently

Our pure-Python implementation is roughly 1000x slower than C implementations
like libsecp256k1. The gap comes from several optimizations we deliberately
omit for educational clarity:

| Optimization | Our Implementation | Production |
|---|---|---|
| **Coordinates** | Affine (modular inversion per addition) | Projective/Jacobian (batch inversions) |
| **Multiplication** | Bit-by-bit scanning | Windowed methods (process 4-5 bits at once) |
| **Generator table** | 256 precomputed points | Larger tables with window combinations |
| **Endomorphism** | None | GLV decomposition (secp256k1-specific) |
| **Constant-time** | No (branches on secret bits) | Yes (side-channel resistant) |
| **Language** | Pure Python | Optimized C/assembly |

The precomputed table (added in the Phase 3 cleanup) closed part of the gap:
test suite runtime dropped from ~75s to ~37s, because most tests multiply by G.

## Results

The precomputed generator table provides a significant speedup by eliminating
all doublings during G-multiplication. But the bigger performance story is
about what we're *not* doing: projective coordinates, windowed methods, and
native code. Those optimizations would close the remaining ~1000x gap to
production libraries.

## Next Steps

- Profile a full signing ceremony to see where time is spent (DKG key
  generation dominates because of many scalar multiplications).
- The `point.py` module documents the precomputed table in detail.
- See Guide Chapter 9 (Architecture) for the full performance discussion.
